In [ ]:
!pip install pm4py
!pip install -q openai>=1.40.0
!pip install -U transformers

from openai import OpenAI
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

import os, re, glob, json, requests, time

system_prompt = open("""/path/to/P1.2_prompt.txt""").read()   # Path to the system prompt file

In [ ]:
DFG_PATH = """/path/to/dfg/files/"""  # Path to the directory containing DFG CSV files
Variants_PATH = """/path/to/variants/files/"""  # Path to the directory containing variants text files
PetriNet_PATH = """/path/to/petrinet/files/"""  # Path to the directory containing Petri net PNML files

MAIN_OUT_PATH = """/path/to/output/results/"""  # Path to save the output JSON file


def sort_filename(name):
    """
    Extract dataset, error type, ratio from filename like:
    DFG_BPIC15-Polluted-0.30.txt
    """
    m = re.match(r"(DFG|Variants|PetriNet)_([A-Za-z0-9]+)-([A-Za-z]+)-([0-9.]+)\.txt", name) # DFG / Variants / PetriNet
    if not m:
        return ("ZZZ", "ZZZ", 999)  # send unknowns to end

    dataset = m.group(1)
    err = m.group(2)
    ratio = float(m.group(3))
    return (dataset, err, ratio)

In [ ]:
# GPT-4o client

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def extract_json(text):
    """Clean model output and extract valid JSON only."""
    if not text or not text.strip():
        raise ValueError("Empty output from model")

    t = text.strip()

    # remove code fences
    t = re.sub(r"```(json)?", "", t)
    t = t.replace("```", "").strip()

    # extract JSON by finding first '{' and last '}'
    start = t.find("{")
    end = t.rfind("}")

    if start == -1 or end == -1:
        raise ValueError(f"No JSON object found in: {t[:300]}...")

    json_str = t[start:end+1]

    return json.loads(json_str)


def safe_llm_query(prompt_messages):
    """Call LLM and handle context-length errors with truncation fallback."""
    try:
        resp = client.chat.completions.create(
            model="gpt-4o",
            temperature=0.1,
            messages=prompt_messages
        )
        return resp.choices[0].message.content

    except Exception as e:
        msg = str(e)
        # Detect OpenAI context-length error
        if "context_length_exceeded" in msg or "maximum context length" in msg:
            # Extract user message content from messages
            user_content = prompt_messages[-1]["content"]

            # Fallback truncation strategy: take head + tail
            MAX_CHARS = 5000
            if len(user_content) > MAX_CHARS * 2:
                truncated = user_content[:MAX_CHARS] + "\n...\n" + user_content[-MAX_CHARS:]
            else:
                truncated = user_content  # already small

            # retry with truncated input
            retry_messages = prompt_messages[:-1] + [
                {"role": "user", "content": truncated}
            ]
            print("Context too large → using truncated DFG for analysis…")

            resp = client.chat.completions.create(
                model="gpt-4o",
                temperature=0.1,
                messages=retry_messages
            )
            return resp.choices[0].message.content
        raise


def analyze_file(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        abstract_text = f.read()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Abstracted process data:\n\n{abstract_text}\n\nReturn JSON only."}
    ]
    resp_text = safe_llm_query(messages)
    return resp_text


raw_results = []
TXT_GLOB = DFG_PATH # or Variants_PATH or PetriNet_PATH depending on which files to analyze

for p in glob.glob(TXT_GLOB, recursive=True):
    basename = os.path.basename(p)
    try:
        model_out = analyze_file(p)
        parsed = extract_json(model_out)
        raw_results.append({"file": basename, "result": parsed})
        print("✅", basename)

    except Exception as e:
        raw_results.append({
            "file": basename,
            "error": str(e),
            "raw": model_out if 'model_out' in locals() else ""
        })
        print("⚠️", basename, "→", e)

sorted_results = sorted(raw_results, key=lambda x: sort_filename(x["file"]))


OUT_PATH = MAIN_OUT_PATH + """your_output_filename.json"""  # Specify your output filename here
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(sorted_results, f, ensure_ascii=False, indent=2)

print("Saved sorted results →", OUT_PATH)

In [ ]:
# GROK-4-fast Client

GROK_API_KEY = """GROK_API_KEY"""
GROK_URL = "https://api.x.ai/v1/chat/completions"
GROK_MODEL = "grok-4-fast"

assert GROK_URL.startswith("https://"), f"ERROR: GROK_URL invalid: {GROK_URL}"
assert GROK_API_KEY.startswith("xai-"), f"ERROR: GROK_API_KEY invalid: {GROK_API_KEY}"


def grok_chat(messages, max_retries=4):
    payload = {
        "model": GROK_MODEL,
        "messages": messages,
        "temperature": 0.1,
        "stream": False
    }
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {GROK_API_KEY}"
    }

    attempt = 0

    while attempt < max_retries:
        resp = requests.post(GROK_URL, headers=headers, json=payload)
        text = resp.text

        # ---------- 1) 429 Too Many Requests → retry ----------
        if resp.status_code == 429:
            wait = 2 ** attempt
            print(f"Grok overloaded (429). Retrying in {wait} seconds…")
            time.sleep(wait)
            attempt += 1
            continue

        # ---------- 2) context-length fallback ----------
        ctx_err = (
            "context" in text.lower()
            or "maximum prompt length" in text.lower()
            or "maximum context" in text.lower()
            or "request contains" in text.lower()
            or "invalid argument" in text.lower()
        )

        if resp.status_code == 400 and ctx_err:
            print("Context too large → applying truncation fallback…")

            original_user = messages[-1]["content"]
            MAX_CHARS = 5000

            if len(original_user) > MAX_CHARS * 2:
                truncated = (
                    original_user[:MAX_CHARS]
                    + "\n...\n"
                    + original_user[-MAX_CHARS:]
                )
            else:
                truncated = original_user

            retry_messages = messages[:-1] + [
                {"role": "user", "content": truncated}
            ]
            payload["messages"] = retry_messages
            resp = requests.post(GROK_URL, headers=headers, json=payload)
            text = resp.text

            if resp.status_code != 200:
                raise RuntimeError(f"Fallback also failed: {text}")

            return resp.json()["choices"][0]["message"]["content"]

        # ---------- 3) Other errors ----------
        if resp.status_code != 200:
            raise RuntimeError(text)

        # ---------- 4) Success ----------
        return resp.json()["choices"][0]["message"]["content"]

    raise RuntimeError("Exceeded retry limit.")


def analyze_file(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        abstract_text = f.read()

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Abstracted process data:\n\n{abstract_text}\n\nReturn JSON only."}
    ]
    return grok_chat(messages)



raw_results = []
TXT_GLOB = DFG_PATH # | Variants_PATH | PetriNet_PATH depending on which files to analyze

for p in glob.glob(TXT_GLOB, recursive=True):
    basename = os.path.basename(p)
    try:
        out_text = analyze_file(p)
        parsed = extract_json(out_text)
        raw_results.append({"file": basename, "result": parsed})
        print("✅", basename)

    except Exception as e:
        raw_results.append({
            "file": basename,
            "error": str(e),
            "raw": out_text if 'out_text' in locals() else ""
        })
        print("⚠️", basename, "→", e)

sorted_results = sorted(raw_results, key=lambda x: sort_filename(x["file"]))

OUT_PATH = MAIN_OUT_PATH + """your_output_filename.json"""  # Specify your output filename here
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(sorted_results, f, ensure_ascii=False, indent=2)

In [ ]:
# LLaMa-3.1-8B-Instruct client

HF_TOKEN = """HuggingFace_API"""
HF_ENDPOINT_URL ="https://gljee5ezsvfuuhzc.us-east-1.aws.endpoints.huggingface.cloud/v1/"

client = OpenAI(
    base_url = HF_ENDPOINT_URL,
    api_key  = HF_TOKEN
)


def safe_chat(messages, model="meta-llama/Llama-3.1-8B-Instruct"):
    """
    Wrapper for HuggingFace Endpoint with fallback.
    """
    try:
        resp = client.chat.completions.create(
            model=model,
            messages=messages,
            temperature=0.1,
            max_tokens=900
        )
        return resp.choices[0].message.content

    except Exception as e:
        msg = str(e).lower()

        # long input / context issues
        if ("context" in msg or "too long" in msg or "maximum" in msg):
            print("Input too large → truncation fallback")

            user_msg = messages[-1]["content"]
            MAX = 6000
            trunc = user_msg[:MAX] + "\n...\n" + user_msg[-MAX:]

            messages = [
                messages[0],
                {"role": "user", "content": trunc}
            ]

            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=0.1,
                max_tokens=900
            )
            return resp.choices[0].message.content

        # rate limiting (429) → exponential retry
        if "429" in msg:
            for wait in [2, 4, 8, 16]:
                print(f"HF endpoint busy → waiting {wait}s ...")
                time.sleep(wait)
                try:
                    resp = client.chat.completions.create(
                        model=model,
                        messages=messages,
                        temperature=0.1,
                        max_tokens=900
                    )
                    return resp.choices[0].message.content
                except:
                    continue

        raise


def analyze_file(txt_path, prompt=system_prompt):
    """
    Pass a single file through Llama 3.1.
    """
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user",   "content": f"Abstracted process:\n\n{content}\n\nReturn JSON only."}
    ]

    resp_text = safe_chat(messages)
    return resp_text


raw_results = []
TXT_GLOB = DFG_PATH # | Variants_PATH | PetriNet_PATH depending on which files to analyze

for p in glob.glob(TXT_GLOB, recursive=True):
    basename = os.path.basename(p)
    try:
        out_text = analyze_file(p)
        parsed = extract_json(out_text)
        raw_results.append({"file": basename, "result": parsed})
        print("✅", basename)

    except Exception as e:
        raw_results.append({
            "file": basename,
            "error": str(e),
            "raw": out_text if 'out_text' in locals() else ""
        })
        print("⚠️", basename, "→", e)

sorted_results = sorted(raw_results, key=lambda x: sort_filename(x["file"]))

OUT_PATH = MAIN_OUT_PATH + """your_output_filename.json"""  # Specify your output filename here
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(sorted_results, f, ensure_ascii=False, indent=2)

In [ ]:
# DeepSeek-V3-0324 Client

USE = "router"   # "pipeline" / "transformers" / "router"
MODEL_NAME = "deepseek-ai/DeepSeek-V3-0324"

pipe = None
tokenizer = None
model = None
client = None

if USE == "pipeline":
    pipe = pipeline(
        "text-generation",
        model=MODEL_NAME,
        trust_remote_code=True
    )

elif USE == "transformers":
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, trust_remote_code=True)
    model = model.to("cuda" if model.device.type == "cuda" else "cpu")

elif USE == "router":
    os.environ['HF_TOKEN'] = """"HuggingFace_API"""
    client = OpenAI(
        base_url="https://router.huggingface.co/v1",
        api_key=os.environ["HF_TOKEN"],
    )


def safe_chat(messages):
    """
    Unified safe chat handler for:
    - pipeline
    - transformers
    - HF Router API

    Includes:
    ✔ Overlength input → truncate fallback
    ✔ 429 rate limit → exponential retry
    ✔ HF context length limitation handling
    """

    def _infer(msgs):
        # pipeline version
        if USE == "pipeline":
            res = pipe(msgs, max_new_tokens=900)
            return res[0]["generated_text"]

        # transformers version
        elif USE == "transformers":
            text_msgs = [
                {"role": m["role"], "content": m["content"]} for m in msgs
            ]
            inputs = tokenizer.apply_chat_template(
                text_msgs,
                add_generation_prompt=True,
                return_tensors="pt"
            ).to(model.device)

            out = model.generate(**inputs, max_new_tokens=900)
            decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:])
            return decoded

        # HF router version
        elif USE == "router":
            resp = client.chat.completions.create(
                model=MODEL_NAME,
                messages=msgs,
                temperature=0.1,
                max_tokens=900
            )
            return resp.choices[0].message.content

    # TRY MAIN INFERENCE
    try:
        return _infer(messages)

    except Exception as e:
        msg = str(e).lower()

        # 1) INPUT TOO LONG → TRUNCATION
        if ("context" in msg or "length" in msg or "too long" in msg):
            print("Input too large → truncation fallback")

            user_text = messages[-1]["content"]
            MAX = 6000
            trunc = user_text[:MAX] + "\n...\n" + user_text[-MAX:]

            new_msg = [
                messages[0],
                {"role": "user", "content": trunc}
            ]
            return _infer(new_msg)

        # 2) 429 RATE LIMIT → RETRY
        if "429" in msg:
            waits = [2, 4, 8, 16]
            for w in waits:
                print(f"HF router busy → waiting {w}s …")
                time.sleep(w)
                try:
                    return _infer(messages)
                except:
                    continue

    raise e


def analyze_file(txt_path, prompt=system_prompt):
    with open(txt_path, "r", encoding="utf-8") as f:
        content = f.read()

    messages = [
        {"role": "system", "content": prompt},
        {"role": "user",   "content": f"Abstracted process:\n\n{content}\n\nReturn JSON only."}
    ]

    resp_text = safe_chat(messages)
    return resp_text


raw_results = []
TXT_GLOB = DFG_PATH # | Variants_PATH | PetriNet_PATH depending on which files to analyze

for p in glob.glob(TXT_GLOB, recursive=True):
    basename = os.path.basename(p)
    try:
        out_text = analyze_file(p)
        parsed = extract_json(out_text)
        raw_results.append({"file": basename, "result": parsed})
        print("✅", basename)

    except Exception as e:
        raw_results.append({
            "file": basename,
            "error": str(e),
            "raw": out_text if 'out_text' in locals() else ""
        })
        print("⚠️", basename, "→", e)

sorted_results = sorted(raw_results, key=lambda x: sort_filename(x["file"]))

OUT_PATH = MAIN_OUT_PATH + """your_output_filename.json"""  # Specify your output filename here
with open(OUT_PATH, "w", encoding="utf-8") as f:
    json.dump(sorted_results, f, ensure_ascii=False, indent=2)